# 10 · Evaluación de Baselines — Sprint 3
**Proyecto:** Detección de Sitios Web Fraudulentos (Phishing)  
**Sprint 3 — PB-11:** Evaluación con métricas de clasificación y cross-validation

---

## Objetivo

Evaluar cada modelo baseline con métricas robustas (F1, AUC-ROC, Recall, Precision)
usando `StratifiedKFold`, y generar visualizaciones diagnósticas: curvas ROC,
matrices de confusión y classification reports.

## Justificación de métricas principales

En detección de phishing:
- **Recall** — prioridad alta: un falso negativo (phishing no detectado) tiene alto costo para el usuario.
- **F1-Score** — métrica principal: equilibrio entre precision y recall, robusta ante clases desbalanceadas.
- **AUC-ROC** — capacidad discriminatoria general del modelo.
- **Accuracy** — referencia, no usada para decisiones (puede ser engañosa con desbalance).

> Esta decisión de métricas está alineada con los criterios de éxito definidos en `01_business.ipynb` (Sprint 1).


## 1. Imports y carga de modelos

In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    PrecisionRecallDisplay,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate

from src.models import get_baseline_models, load_model

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
print('✓ Librerías cargadas.')

## 2. Cargar datos y modelos

In [ ]:
TARGET_COL = 'Result'

# Train (crudos, con SMOTE)
train_df = pd.read_csv('../data/processed/train.csv')
X_train = train_df.drop(columns=[TARGET_COL])
y_train = train_df[TARGET_COL]

# Test (crudos)
test_df = pd.read_csv('../data/processed/test.csv')
X_test = test_df.drop(columns=[TARGET_COL])
y_test = test_df[TARGET_COL]

print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Distribución y_train: {dict(y_train.value_counts().sort_index())}')
print(f'Distribución y_test:  {dict(y_test.value_counts().sort_index())}')

In [ ]:
# Cargar todos los modelos entrenados en 09_baseline_models.ipynb
model_names = list(get_baseline_models().keys())

models = {}
for name in model_names:
    short = name.lower().replace(' ', '_')
    try:
        models[name] = load_model(short, model_dir='../models')
    except FileNotFoundError:
        print(f'⚠️  Modelo no encontrado: {name}. Ejecutar primero 09_baseline_models.ipynb')

print(f'\n✅ Modelos cargados: {list(models.keys())}')

## 3. Cross-validation estratificada (5-fold)

Se usa `StratifiedKFold` para preservar la proporción de clases en cada fold.
Se evalúan 5 métricas en train y test para detectar overfitting.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

SCORING = {
    'accuracy':  'accuracy',
    'f1':        'f1',
    'precision': 'precision',
    'recall':    'recall',
    'roc_auc':   'roc_auc',
}

cv_results = {}
for name, pipe in models.items():
    print(f'CV: {name}...')
    res = cross_validate(pipe, X_train, y_train, cv=cv,
                         scoring=SCORING, return_train_score=True, n_jobs=-1)
    cv_results[name] = res
    print(f'  F1={res["test_f1"].mean():.4f} | AUC={res["test_roc_auc"].mean():.4f} | Recall={res["test_recall"].mean():.4f}')

print('\n✅ Cross-validation completa.')

## 4. Classification Report por modelo

In [ ]:
class_names = ['Phishing (-1)', 'Legítimo (1)']

print('=' * 70)
for name, pipe in models.items():
    y_pred = pipe.predict(X_test)
    print(f'\n--- {name} ---')
    print(classification_report(y_test, y_pred, target_names=class_names))
    print('=' * 70)

## 5. Matrices de confusión

In [ ]:
n = len(models)
ncols = 4
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 3.5))
axes = axes.ravel()

for i, (name, pipe) in enumerate(models.items()):
    ConfusionMatrixDisplay.from_estimator(
        pipe, X_test, y_test,
        display_labels=class_names,
        ax=axes[i], cmap='Blues', colorbar=False
    )
    axes[i].set_title(name, fontsize=10, fontweight='bold')
    axes[i].set_xlabel('Predicho', fontsize=8)
    axes[i].set_ylabel('Real', fontsize=8)

# Ocultar ejes sobrantes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Matrices de Confusión — Baselines (Test Set)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../models/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Curvas ROC

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

palette = plt.cm.tab10.colors
for i, (name, pipe) in enumerate(models.items()):
    RocCurveDisplay.from_estimator(
        pipe, X_test, y_test,
        name=name, ax=ax, color=palette[i]
    )

ax.plot([0, 1], [0, 1], 'k--', label='Azar (AUC=0.50)', linewidth=1)
ax.set_title('Curvas ROC — Baselines (Test Set)', fontsize=13, fontweight='bold')
ax.set_xlabel('Tasa de Falsos Positivos', fontsize=11)
ax.set_ylabel('Tasa de Verdaderos Positivos (Recall)', fontsize=11)
ax.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.savefig('../models/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Curvas Precision-Recall

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

for i, (name, pipe) in enumerate(models.items()):
    PrecisionRecallDisplay.from_estimator(
        pipe, X_test, y_test,
        name=name, ax=ax, color=palette[i]
    )

ax.set_title('Curvas Precision-Recall — Baselines (Test Set)', fontsize=13, fontweight='bold')
ax.legend(loc='lower left', fontsize=8)
plt.tight_layout()
plt.savefig('../models/precision_recall_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Detección de overfitting y underfitting

Se compara F1 en train vs. test (CV):
- **Overfitting:** gap > 0.10 (train mucho mejor que test)
- **Underfitting:** F1 test < 0.70 (rendimiento pobre en ambos)

In [ ]:
print(f'{'Modelo':30s} | {'Train F1':>9} | {'Test F1':>9} | {'Gap':>7} | Diagnóstico')
print('-' * 75)

for name, res in cv_results.items():
    train_f1 = res['train_f1'].mean()
    test_f1  = res['test_f1'].mean()
    gap = train_f1 - test_f1

    if gap > 0.10:
        diag = '⚠️  Overfitting'
    elif test_f1 < 0.70:
        diag = '⬇️  Underfitting'
    else:
        diag = '✅ OK'

    print(f'{name:30s} | {train_f1:9.4f} | {test_f1:9.4f} | {gap:7.4f} | {diag}')

## 9. Resumen del notebook — PB-11

### Hallazgos principales

- **Métrica principal elegida:** F1-Score, por el costo asimétrico de falsos negativos en phishing.
- Las curvas ROC permiten seleccionar el umbral de clasificación óptimo en Sprint 4.
- Los modelos con alto Overfit_Gap serán candidatos a regularización en Sprint 4.
- Las matrices de confusión revelan el tipo de error predominante por modelo.

### Archivos generados

| Archivo | Descripción |
|---------|-------------|
| `models/confusion_matrices.png` | Matrices de confusión (test set) |
| `models/roc_curves.png` | Curvas ROC comparativas |
| `models/precision_recall_curves.png` | Curvas Precision-Recall |

### Siguiente paso

→ `11_model_comparison.ipynb` (PB-12): comparación final y selección de candidatos para Sprint 4.
